# DME Express — Delivery Install-Time Model

**Version v1.0.0 — 2026-08-18**

Builds a predictive model for **how long a delivery SHOULD take** from the products it
contains, then rebuilds the **expected completion time for every delivery ticket** year to
date. One SQL row = one product line (one asset) on an order; a full delivery is the set of
lines sharing an **order number + delivery address**. On-site time is
`Completion_Time − Arrival_Time`.

**Method in one line:** expected minutes = fixed per-visit base + Σ (units of product ×
per-unit install minutes), fitted with **non-negative least squares** (NNLS) so no product
can be assigned a physically meaningless negative install time, with **bootstrap confidence
intervals** and a **single-product-delivery median cross-check**.

**Changelog v1.0.0 — 2026-08-18**

1. Initial release. SQL connectivity (Key Vault login, `run_query` read-only guard) lifted
   verbatim from `ops_dashboard v1.8.0` Cells 4.1/4.2 — re-sync if the dashboard changes.
2. **PHI handling — deliberate deviation from the source query.** The analyst-provided
   query selects `Patient_Firstname`, `Patient_Lastname` and `Ship_To_Address`. Patient
   names are **not selected at all** (they play no role in the analysis), and the delivery
   address is **hashed to a SHA2-256 token inside SQL Server** (`HASHBYTES`) so the raw
   address never enters the DataFrame, any output, or this repository. The token is only
   used to distinguish two deliveries that share an order number (CLAUDE.md rule 2).
3. Order numbers are record-level patient references: they appear in **files exported to
   the OneDrive report folder only**, never in notebook output (ops_dashboard v1.3.0
   audit C1 precedent).

**Data limitations (state these when quoting results)**

* `Arrival_Time`/`Completion_Time` are **nvarchar(50)** in two mixed formats — US
  `MM/DD/YYYY HH:MM` (24h) and ISO `datetime2` — parsed with an explicit dual-style
  `TRY_CONVERT` (probed 2026-08-18: 0 residual unparseable in the delivery population).
* **`SERP_ORDERS_HISTORY` lags: at first run its delivery data ends 2025-11.** A
  "current year to date" window is therefore empty; when that happens the notebook falls
  back — loudly — to Jan-1 → max-date of the **latest year that has data**, and files the
  gap as research item R0. Where current-year orders live is an open question for IT.
* `Arrival_Time`/`Completion_Time` semantics are assumed to bound the on-site visit. If
  either is entered at batch close-out rather than in the field, its delivery is not
  usable — the bulk-timestamp detector (Cell 9) exists for exactly this and reports what
  it excludes rather than silently dropping it.
* Timestamps that fail `TRY_CONVERT(DATETIME, …)` are excluded and **counted** by the
  probe in Cell 6.2; a nonzero count is an upstream feed problem, not noise.
* The model prices the **marginal product mix**, not travel, and not technician skill.
  A warehouse whose actual/expected ratio sits above 1 may have harder addresses, not
  slower technicians.

**Attribution caveat (repeat in anything published from this notebook):** expected times
are modeled averages — **proximity, not fault**. Suitable for planning, staffing and
coaching conversations; never discipline without ticket-level review.


## Cell 1 — INSTALL DEPENDENCIES

In [ ]:
%pip install matplotlib python-dotenv pypyodbc openpyxl python-dateutil azure-identity azure-keyvault-secrets scipy

## Cell 2 — IMPORTS *(LIFTED from ops_dashboard v1.8.0 Cell 2, trimmed to what this notebook uses; scipy added for NNLS)*

In [ ]:
# Standard library + analysis stack. azure.identity/azure.keyvault power the Key
# Vault login in Cell 4. scipy.optimize.nnls is the reviewed non-negative least
# squares implementation — do not hand-roll Lawson–Hanson here.
import os, re, time, warnings
from datetime import datetime, date, timedelta
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pypyodbc as odbc
from scipy.optimize import nnls
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')
CHART_DPI = 150
matplotlib.rcParams['figure.dpi'] = CHART_DPI
print(f'pandas {pd.__version__}  |  numpy {np.__version__}')

## Cell 3 — RUN WINDOW, PATHS & MODEL CONFIGURATION

*Teaching note:* every threshold an analyst might reasonably want to move lives here, not
buried in the cells that use it. When the CFO asks "what if the cap were 6 hours instead of
8?", the answer should be a one-line change and a re-run, not a search.

In [ ]:
RUN_DATE     = datetime.now().strftime('%Y-%m-%d')
FILTER_START = '2026-01-01'                       # current year to date
AS_OF_DATE   = datetime.now().date() - timedelta(days=1)   # last full day of data
FILTER_END   = AS_OF_DATE.strftime('%Y-%m-%d')
FALLBACK_TO_LATEST_YTD = True   # SERP_ORDERS_HISTORY lags (probed: ends 2025-11). If the
                                # requested window is EMPTY, fall back to Jan-1 -> max-date
                                # of the latest year with data, and file the gap as R0.
print(f'Requested window: {FILTER_START} -> {FILTER_END}')

# Deliverables go OUTSIDE the repository (ops_dashboard v1.8.0 audit C3: a relative
# deliverable path re-creates the '$HOME' incident). Refuse a relative root outright.
USER_ROOT = os.path.expanduser('~')
assert os.path.isabs(USER_ROOT), f'USER_ROOT must be absolute, got {USER_ROOT!r}'
REPORT_ROOT = f'{USER_ROOT}/OneDrive - DME Express/Reports/InstallTimeModel'
OUT_DIR = os.path.join(REPORT_ROOT, RUN_DATE)
assert os.path.isabs(OUT_DIR), f'OUT_DIR must be absolute, got {OUT_DIR!r}'
os.makedirs(OUT_DIR, exist_ok=True)
print(f'Output: {OUT_DIR}')

DRIVER_NAME   = 'ODBC Driver 18 for SQL Server'
SERVER_NAME   = 'tcp:dmeexpress.database.windows.net,1433'
DATABASE_NAME = 'DMEEXPRESS'

# ── Delivery-duration quality gates ──────────────────────────────────────────
MIN_DURATION_MIN    = 3.0     # below this a visit cannot physically have happened
MAX_DURATION_MIN    = 480.0   # above 8h the timestamps describe something else
SPLIT_TOLERANCE_MIN = 60.0    # completion spread across one order's lines beyond this
                              # = split/multi-trip delivery, not one visit
BULK_TS_MIN_DELIVERIES = 12   # >= this many DELIVERIES sharing one exact completion
                              # timestamp = batch close-out, not field data

# ── Model configuration ──────────────────────────────────────────────────────
MIN_DELIVERIES_PER_PRODUCT = 30           # support floor for a product to get its own
                                          # coefficient; below it -> OTHER bucket
OTHER_LABEL = 'OTHER (below support floor)'
N_BOOT     = 200                          # bootstrap resamples for coefficient CIs
BOOT_SEED  = 20260818

# ── Anomaly / research thresholds ────────────────────────────────────────────
SLOW_RATIO = 2.5      # actual > 2.5x expected  -> ticket-level review
FAST_RATIO = 0.4      # actual < 0.4x expected  -> ticket-level review
XCHECK_DIVERGENCE_PCT = 50.0   # model vs single-product-median divergence flags
XCHECK_DIVERGENCE_MIN = 15.0   # ... only when it also exceeds this many minutes
TOP_N_RESEARCH = 200  # per category, exported for review

PALETTE = {'primary': '#1f77b4', 'muted': '#9467bd', 'ink': '#000000'}  # ops dashboard hues

def save_fig(fig, name):
    fig.savefig(os.path.join(OUT_DIR, f'{name}_{RUN_DATE}.png'), bbox_inches='tight', dpi=CHART_DPI)

## Cell 4 — SECURE SQL CONNECTION: AZURE KEY VAULT *(LIFTED VERBATIM from ops_dashboard v1.8.0 Cell 4.1)*

In [ ]:
KEY_VAULT_URI   = 'https://dmee-keyvault.vault.azure.net/'
HOSTNAME_SECRET = 'DataWarehouseHostname'
SQL_USERNAME    = 'anewton-ro'   # secret name is the login; its value is the password

kv = SecretClient(vault_url=KEY_VAULT_URI, credential=DefaultAzureCredential())
# v1.3.0 (audit B4): the vault stores a BARE hostname. Substituting it for the
# 'tcp:host,1433' form produced '[08001] TCP Provider: Timeout error [258]' on a cold
# connect; wrapping it connects first try. Keep the prefix and the port.
SERVER_NAME  = f'tcp:{kv.get_secret(HOSTNAME_SECRET).value},1433'
SQL_PASSWORD = kv.get_secret(SQL_USERNAME).value
print('Secrets loaded from Key Vault.')

sql_conn = odbc.connect(
    f'DRIVER={{{DRIVER_NAME}}};SERVER={SERVER_NAME};DATABASE={DATABASE_NAME};'
    f'UID={SQL_USERNAME};PWD={SQL_PASSWORD};'
    f'Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;'
)
print('DB connection established.')

## Cell 5 — QUERY RUNNER *(LIFTED VERBATIM from ops_dashboard v1.8.0 Cell 4.2 — the in-process read-only guard required by CLAUDE.md rule 4)*

In [ ]:
_READ_ONLY_STARTS = ('select', 'with')
def run_query(query, label='', verbose=False):
    """Run a READ-ONLY query and return a DataFrame.

    v1.3.0 (audit C2): CLAUDE.md rule 4 says analysis queries are SELECT/WITH only and
    that this helper enforces it in-process. Anything that is not a single SELECT/WITH
    statement raises before touching the connection. This is a guard against an accident
    in a notebook that holds a live production connection, not a security boundary.
    """
    _q = re.sub(r'--[^\n]*', ' ', query)            # strip line comments
    _q = re.sub(r'/\*.*?\*/', ' ', _q, flags=re.S)  # strip block comments
    _q_clean = _q.strip().lstrip('(').lstrip()
    if not _q_clean.lower().startswith(_READ_ONLY_STARTS):
        raise ValueError(f'run_query is read-only: a query must begin with SELECT or WITH '
                         f'(label={label!r}, got {_q_clean[:60]!r}).')
    _forbidden = re.findall(r'(?<![\w.])(insert|update|delete|merge|drop|truncate|alter|create|'
                            r'grant|revoke|exec|execute|sp_\w+|xp_\w+)(?![\w.])', _q, flags=re.I)
    if _forbidden:
        raise ValueError(f'run_query is read-only: refusing statement containing '
                         f'{sorted(set(w.lower() for w in _forbidden))} (label={label!r}).')
    if ';' in _q_clean.rstrip().rstrip(';'):
        raise ValueError(f'run_query is read-only: one statement per call (label={label!r}).')
    if verbose: print(f'Query: {label}\n{query}')
    t0 = time.time()
    cur = sql_conn.cursor(); cur.execute(query)
    rows = cur.fetchall()
    cols = [c[0].lower() for c in cur.description]
    df   = pd.DataFrame(rows, columns=cols)
    if label: print(f'  {label}: {len(df):,} rows  ({time.time()-t0:.1f}s)')
    return df

## Cell 6.1 — SCHEMA PROBE: SERP_ORDERS_HISTORY

*Teaching note:* this table is new to the repo's data dictionary, so the first thing the
notebook does is ask the database what the columns actually are — in particular whether
`Arrival_Time`/`Completion_Time` are real datetimes or varchar. Assuming a type instead of
probing it is how the tech_workload notebook once dropped an entire day of tickets
(v1.33.0 H3). Column *names and types* contain no patient data, so printing this is safe.

In [ ]:
_schema = run_query("""
SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_NAME = 'SERP_ORDERS_HISTORY'
ORDER BY ORDINAL_POSITION
""", 'SERP_ORDERS_HISTORY schema')
print(_schema.to_string(index=False))

## Cell 6.2 — DATA AVAILABILITY, ORDERS EXTRACT (PHI-SAFE) + TIMESTAMP PROBE

The analyst's source query, with three deliberate changes (see header changelog):
patient name columns removed, `Ship_To_Address` hashed to `addr_token` **inside SQL
Server**, and the window bounded on the converted completion timestamp.

*Teaching note:* the schema probe shows both time columns are `nvarchar(50)`, and sampling
found TWO formats mixed in one column: US `MM/DD/YYYY HH:MM` and ISO
`yyyy-mm-dd hh:mm:ss.0000000`. A bare `TRY_CONVERT(DATETIME, …)` NULLs the ISO variant
(7-digit fractions overflow DATETIME) and leaves the US variant at the mercy of the
login's language setting. The dual-style expression below is explicit about both, so the
parse result cannot change because someone's session language did. The availability probe
runs FIRST because a lagging history table makes a "year to date" window silently empty —
zero rows must be a stated finding, not a mystery.

In [ ]:
# Explicit dual-format timestamp parse: style 101 = MM/DD/YYYY, style 121 = ISO.
_TS = "COALESCE(TRY_CONVERT(DATETIME2(0), {c}, 101), TRY_CONVERT(DATETIME2(0), {c}, 121))"
_CP, _AR = _TS.format(c='O.Completion_Time'), _TS.format(c='O.Arrival_Time')
_BASE_FILTERS = """O.Asset_Tag IS NOT NULL
  AND O.[Status] <> 'Canceled'
  AND LEFT(O.Asset_Tag, 2) <> '99'
  AND O.Asset_Tag NOT LIKE '%ATNR%'
  AND O.Order_Type = 'Delivery'
  AND O.Reason_For NOT IN ('Billing Switch')"""

# ── Availability probe: does the requested window exist in this table at all? ──
_avail = run_query(f"""
SELECT CONVERT(VARCHAR(10), MIN({_CP}), 120) AS min_dt,
       CONVERT(VARCHAR(10), MAX({_CP}), 120) AS max_dt,
       SUM(CASE WHEN {_CP} >= '{FILTER_START}' AND {_CP} < DATEADD(day, 1, CONVERT(DATE, '{FILTER_END}'))
                THEN 1 ELSE 0 END) AS lines_in_window,
       SUM(CASE WHEN {_CP} IS NULL AND O.Completion_Time IS NOT NULL
                 AND LTRIM(RTRIM(O.Completion_Time)) <> '' THEN 1 ELSE 0 END) AS unparseable
FROM dbo.[SERP_ORDERS_HISTORY] AS O WITH (NOLOCK)
WHERE {_BASE_FILTERS}
""", 'Availability probe')
_min_dt, _max_dt = str(_avail.iloc[0, 0]), str(_avail.iloc[0, 1])
_in_window = int(_avail.iloc[0, 2] or 0)
UNPARSEABLE_COMPLETION_ROWS = int(_avail.iloc[0, 3] or 0)
print(f'  table coverage: {_min_dt} -> {_max_dt}   lines in requested window: {_in_window:,}')
if UNPARSEABLE_COMPLETION_ROWS:
    print(f'  *** {UNPARSEABLE_COMPLETION_ROWS:,} delivery lines have UNPARSEABLE '
          f'Completion_Time even under the dual-format parse -> research queue (R1).')

WINDOW_FELL_BACK = False
EFFECTIVE_START, EFFECTIVE_END = FILTER_START, FILTER_END
if _in_window == 0 and FALLBACK_TO_LATEST_YTD:
    WINDOW_FELL_BACK = True
    EFFECTIVE_START = f'{_max_dt[:4]}-01-01'
    EFFECTIVE_END   = _max_dt
    print(f'\n  *** REQUESTED WINDOW IS EMPTY: SERP_ORDERS_HISTORY ends {_max_dt}. ***')
    print(f'  *** Falling back to the latest available year to date: '
          f'{EFFECTIVE_START} -> {EFFECTIVE_END}. Filed as research item R0 —          ***')
    print(f'  *** where do current-year delivery orders live? Ask IT.                  ***')
assert _in_window > 0 or WINDOW_FELL_BACK, 'No data in window and fallback disabled.'

print(f'\nExtracting deliveries ({EFFECTIVE_START} to {EFFECTIVE_END})...')
df_lines = run_query(f"""
SELECT
    TRIM(O.[Order])                        AS order_num,
    TRIM(ISNULL(O.ProductName, ''))        AS product_name,
    TRIM(ISNULL(O.Asset_Tag, ''))          AS asset_tag,
    TRIM(ISNULL(O.Warehouse, ''))          AS warehouse,
    CONVERT(VARCHAR(32), HASHBYTES('SHA2_256',
        UPPER(LTRIM(RTRIM(ISNULL(O.Ship_To_Address, ''))))), 2) AS addr_token,
    {_AR} AS arrival_dt,
    {_CP} AS completion_dt
FROM dbo.[SERP_ORDERS_HISTORY] AS O WITH (NOLOCK)
WHERE {_BASE_FILTERS}
  AND {_CP} >= '{EFFECTIVE_START}'
  AND {_CP} < DATEADD(day, 1, CONVERT(DATE, '{EFFECTIVE_END}'))
OPTION (RECOMPILE, MAXDOP 4)
""", 'Order lines')

# Shape/null summary only — order numbers and tokens are record-level patient references
# and never render in notebook output (ops_dashboard v1.3.0 audit C1 precedent).
print(f'  df_lines: {len(df_lines):,} rows x {df_lines.shape[1]} cols')
print('  nulls by column: ' + (', '.join(f'{c}={int(n)}' for c, n in df_lines.isna().sum().items() if n) or 'none'))
for c in ('arrival_dt', 'completion_dt'):
    df_lines[c] = pd.to_datetime(df_lines[c], errors='coerce')
    print(f"  {c} range: {df_lines[c].min()} -> {df_lines[c].max()}")

## Cell 7 — LINE PREP: PRODUCT NORMALISATION & DEDUPLICATION

*Teaching note:* free-text product names drift ('CONCENTRATOR ', 'Concentrator'). Model
features key on an uppercased, whitespace-collapsed `product_norm`; a display map keeps
the most common original spelling for outputs. Deduplication keys on
(order, address token, asset tag) — the same physical asset cannot be delivered twice on
one visit, so a repeat is a data-entry echo, and echoes inflate that product's apparent
quantity and bias its install time downward.

In [ ]:
df_lines['product_norm'] = (df_lines['product_name'].astype(str)
                            .str.upper().str.replace(r'\s+', ' ', regex=True).str.strip())
df_lines = df_lines[df_lines['product_norm'] != ''].copy()

# Most common original spelling per normalised key, for report output.
PRODUCT_DISPLAY = (df_lines.groupby('product_norm')['product_name']
                   .agg(lambda s: s.mode().iat[0]).to_dict())

_pre = len(df_lines)
df_lines = df_lines.drop_duplicates(subset=['order_num', 'addr_token', 'asset_tag'])
DUP_LINES_REMOVED = _pre - len(df_lines)
print(f'  duplicate (order, address, asset) echoes removed: {DUP_LINES_REMOVED:,}')

df_lines['delivery_key'] = df_lines['order_num'].astype(str) + '|' + df_lines['addr_token'].astype(str)
print(f'  lines: {len(df_lines):,}   distinct deliveries: {df_lines["delivery_key"].nunique():,}   '
      f'distinct products: {df_lines["product_norm"].nunique():,}')

## Cell 8 — CONSOLIDATE ORDER LINES → DELIVERY EVENTS

One delivery = all lines sharing (order number, address token). The visit window is
min(arrival) → max(completion) across the lines; the spread between line completion times
is kept as a diagnostic — a wide spread means the "one visit" assumption is false for that
order (split delivery), and it is gated out in Cell 9 rather than averaged over.

*Teaching note:* if the timestamps turn out to be time-of-day only (date part 1900-01-01),
a delivery that crosses midnight computes negative. The wrap below adds 24h to those and
counts them — a modelling decision that must be visible, not an `abs()` hidden in a
formula.

In [ ]:
gb = df_lines.groupby(['delivery_key', 'order_num', 'addr_token'], as_index=False)
df_dlv = gb.agg(
    warehouse       = ('warehouse', lambda s: s.mode().iat[0] if len(s.mode()) else ''),
    n_warehouses    = ('warehouse', 'nunique'),
    n_lines         = ('asset_tag', 'size'),
    n_products      = ('product_norm', 'nunique'),
    arrival_dt      = ('arrival_dt', 'min'),
    completion_dt   = ('completion_dt', 'max'),
    completion_last = ('completion_dt', 'max'),
    completion_first= ('completion_dt', 'min'),
)
df_dlv['completion_spread_min'] = ((df_dlv['completion_last'] - df_dlv['completion_first'])
                                   .dt.total_seconds() / 60.0)
df_dlv['duration_min'] = ((df_dlv['completion_dt'] - df_dlv['arrival_dt'])
                          .dt.total_seconds() / 60.0)

# Time-of-day-only detection: if the date part is overwhelmingly 1900-01-01, the source
# stores clock times, not datetimes. Negative durations then mean a midnight crossing.
_has_arr = df_dlv['arrival_dt'].notna()
TIME_ONLY_MODE = bool(_has_arr.any() and
                      (df_dlv.loc[_has_arr, 'arrival_dt'].dt.year == 1900).mean() > 0.90)
N_MIDNIGHT_WRAPPED = 0
if TIME_ONLY_MODE:
    _wrap = df_dlv['duration_min'].lt(0) & df_dlv['duration_min'].gt(-1440)
    N_MIDNIGHT_WRAPPED = int(_wrap.sum())
    df_dlv.loc[_wrap, 'duration_min'] += 1440.0
    print(f'  TIME-ONLY timestamps detected: {N_MIDNIGHT_WRAPPED:,} negative durations '
          f'wrapped +24h (midnight crossings). LIMITATION: same-clock-time errors are undetectable.')
else:
    print('  Timestamps carry real dates (no midnight wrap applied).')

print(f'  deliveries: {len(df_dlv):,}')
print('  duration_min distribution (pre-QC): '
      + ', '.join(f'P{int(q*100)}={df_dlv["duration_min"].quantile(q):.1f}'
                  for q in (0.25, 0.50, 0.75, 0.95, 0.99)))
print(f'  lines per delivery: P25={df_dlv["n_lines"].quantile(.25):.0f} '
      f'median={df_dlv["n_lines"].median():.0f} P75={df_dlv["n_lines"].quantile(.75):.0f} '
      f'max={df_dlv["n_lines"].max():.0f}')

## Cell 9 — QUALITY GATES, EXCLUSION LEDGER & BULK-EVENT DETECTION

*Teaching note (why a ledger):* every delivery excluded from the model is assigned exactly
one reason and **counted out loud**. The standing rule from the lost-equipment work
applies here too: one-off bulk data events are excluded from metrics and noted separately
— never silently dropped. A batch close-out (many deliveries "completing" at one exact
second) would otherwise teach the model that whole days of work take zero minutes.

In [ ]:
df_dlv['qc'] = 'ok'
def _gate(mask, reason):
    hit = mask & df_dlv['qc'].eq('ok')
    df_dlv.loc[hit, 'qc'] = reason
    return int(hit.sum())

# Bulk-entry detection FIRST (on the consolidated grain): an exact completion timestamp
# shared by many different deliveries is an administrative event, not field work.
_ts_counts = df_dlv.groupby('completion_dt')['delivery_key'].transform('nunique')
_bulk_mask = df_dlv['completion_dt'].notna() & (_ts_counts >= BULK_TS_MIN_DELIVERIES)
_ledger = []
_ledger.append(('bulk_timestamp_cluster', _gate(_bulk_mask, 'bulk_timestamp_cluster')))
_ledger.append(('missing_timestamp',  _gate(df_dlv['arrival_dt'].isna() | df_dlv['completion_dt'].isna(), 'missing_timestamp')))
_ledger.append(('nonpositive_duration', _gate(df_dlv['duration_min'] <= 0, 'nonpositive_duration')))
_ledger.append(('below_min_duration', _gate(df_dlv['duration_min'] < MIN_DURATION_MIN, 'below_min_duration')))
_ledger.append(('above_max_duration', _gate(df_dlv['duration_min'] > MAX_DURATION_MIN, 'above_max_duration')))
_ledger.append(('split_delivery',     _gate(df_dlv['completion_spread_min'] > SPLIT_TOLERANCE_MIN, 'split_delivery')))
_ledger.append(('multi_warehouse',    _gate(df_dlv['n_warehouses'] > 1, 'multi_warehouse')))

_n = len(df_dlv)
print('EXCLUSION LEDGER (one reason per delivery, applied in order):')
for reason, cnt in _ledger:
    print(f'  {reason:<24s} {cnt:7,}  ({cnt/_n*100:5.2f}%)')
N_OK = int(df_dlv['qc'].eq('ok').sum())
print(f'  {"MODEL SAMPLE (ok)":<24s} {N_OK:7,}  ({N_OK/_n*100:5.2f}%)')

# Bulk clusters, noted separately (dates + warehouses are aggregates — safe to print).
if _bulk_mask.any():
    _bk = (df_dlv[_bulk_mask].assign(d=df_dlv['completion_dt'].dt.date)
           .groupby(['d', 'warehouse'])['delivery_key'].nunique()
           .sort_values(ascending=False).head(10))
    print('\nBULK TIMESTAMP CLUSTERS (top 10 by deliveries) — excluded and noted, per the')
    print('bulk-event rule; the underlying tickets go to the research queue (R5):')
    for (d, wh), c in _bk.items():
        print(f'  {d}  {wh:<28s} {c:5,} deliveries')

df_model = df_dlv[df_dlv['qc'] == 'ok'].copy()
assert len(df_model) > 0, 'QC removed every delivery — check the gates before modelling.'
print('\nduration_min distribution (model sample): '
      + ', '.join(f'P{int(q*100)}={df_model["duration_min"].quantile(q):.1f}'
                  for q in (0.25, 0.50, 0.75)))

## Cell 10 — PRODUCT FEATURE MATRIX

*Teaching note (why a support floor):* a product that appears on 4 deliveries all YTD
gives the regression 4 equations to price it with — the estimate would be noise wearing a
number. Products below `MIN_DELIVERIES_PER_PRODUCT` are folded into one OTHER bucket that
absorbs their average burden without pretending we know each one's install time. They are
listed in the research queue (R7) so high-value rare products can be studied directly.

In [ ]:
_lines_ok = df_lines[df_lines['delivery_key'].isin(df_model['delivery_key'])]
qty = (_lines_ok.groupby(['delivery_key', 'product_norm']).size()
       .unstack(fill_value=0))

support = (qty > 0).sum(axis=0).sort_values(ascending=False)   # deliveries containing product
modeled_products = support[support >= MIN_DELIVERIES_PER_PRODUCT].index.tolist()
rare_products    = support[support <  MIN_DELIVERIES_PER_PRODUCT].index.tolist()

X_df = qty[modeled_products].copy()
X_df[OTHER_LABEL] = qty[rare_products].sum(axis=1) if rare_products else 0
X_df = X_df.reindex(df_model.set_index('delivery_key').index, fill_value=0)

feature_names = list(X_df.columns)
X = np.column_stack([np.ones(len(X_df)), X_df.to_numpy(dtype=float)])   # col 0 = base time
y = df_model.set_index('delivery_key').loc[X_df.index, 'duration_min'].to_numpy(dtype=float)

_rare_share = (qty[rare_products].sum().sum() / max(qty.sum().sum(), 1) * 100) if rare_products else 0.0
print(f'  modeled products: {len(modeled_products)}   folded into OTHER: {len(rare_products)} '
      f'({_rare_share:.1f}% of all units)')
print(f'  X: {X.shape[0]:,} deliveries x {X.shape[1]} columns (incl. base-time intercept)')

## Cell 11 — FIT: NON-NEGATIVE LEAST SQUARES + BOOTSTRAP CIs

*Teaching note (why NNLS, not plain OLS):* products travel together (a concentrator
usually ships with tubing), and under that collinearity OLS happily prices one product
negative and its companion too high — the pair still sums right, but "installing tubing
saves 11 minutes" is not a sentence anyone should take to the CFO. NNLS constrains every
install time (and the per-visit base) to ≥ 0, which is the physics of the problem. The
price of the constraint is that genuinely-zero-cost products pile up AT zero — those are
flagged (R6), not trusted. CIs come from a bootstrap because NNLS has no closed-form
standard errors at the boundary.

In [ ]:
t0 = time.time()
coef, _rnorm = nnls(X, y)
yhat = X @ coef
resid = y - yhat
SS_RES = float((resid ** 2).sum()); SS_TOT = float(((y - y.mean()) ** 2).sum())
R2  = 1 - SS_RES / SS_TOT if SS_TOT > 0 else np.nan
MAE = float(np.abs(resid).mean())
BASE_MIN = float(coef[0])
print(f'  fit in {time.time()-t0:.1f}s   R2={R2:.3f}   MAE={MAE:.1f} min   '
      f'base (per-visit) time={BASE_MIN:.1f} min')

rng = np.random.default_rng(BOOT_SEED)
_boot = np.empty((N_BOOT, len(coef)))
t0 = time.time()
for b in range(N_BOOT):
    idx = rng.integers(0, len(y), len(y))
    _boot[b], _ = nnls(X[idx], y[idx])
ci_lo = np.percentile(_boot, 2.5, axis=0)
ci_hi = np.percentile(_boot, 97.5, axis=0)
print(f'  bootstrap: {N_BOOT} resamples in {time.time()-t0:.1f}s   '
      f'base time 95% CI [{ci_lo[0]:.1f}, {ci_hi[0]:.1f}] min')

## Cell 12 — PER-PRODUCT INSTALL-TIME STANDARDS + INDEPENDENT CROSS-CHECK

*Teaching note (why the cross-check):* deliveries containing exactly one product line are
a model-free measurement of that product: their whole duration is base + one install. If
the regression and that simple median disagree badly, one of them is wrong — usually the
regression, dragged by which OTHER products the item co-travels with. Product names are
operational vocabulary, not PHI, so this table prints.

In [ ]:
tbl_products = pd.DataFrame({
    'product_norm': feature_names,
    'install_min':  coef[1:],
    'ci_lo':        ci_lo[1:],
    'ci_hi':        ci_hi[1:],
    'n_deliveries': [int(support.get(p, 0)) if p != OTHER_LABEL else int((X_df[OTHER_LABEL] > 0).sum())
                     for p in feature_names],
    'n_units':      X_df.sum(axis=0).values.astype(int),
})
tbl_products['product'] = [PRODUCT_DISPLAY.get(p, p) for p in tbl_products['product_norm']]

# Cross-check from single-product deliveries with exactly one unit.
_single = df_model[(df_model['n_products'] == 1) & (df_model['n_lines'] == 1)]
_single_prod = (_lines_ok[_lines_ok['delivery_key'].isin(_single['delivery_key'])]
                .set_index('delivery_key')['product_norm'])
_sm = (_single.set_index('delivery_key')
       .assign(product_norm=_single_prod)
       .groupby('product_norm')['duration_min'].agg(['median', 'count']))
tbl_products['single_median_total'] = tbl_products['product_norm'].map(_sm['median'])
tbl_products['single_n']            = tbl_products['product_norm'].map(_sm['count']).fillna(0).astype(int)
tbl_products['model_single_total']  = BASE_MIN + tbl_products['install_min']
_div = (tbl_products['model_single_total'] - tbl_products['single_median_total'])
tbl_products['xcheck_gap_min'] = _div
tbl_products['flag_zero_boundary'] = tbl_products['install_min'] < 0.5
tbl_products['flag_xcheck'] = (tbl_products['single_n'] >= 10) & (
    _div.abs() > XCHECK_DIVERGENCE_MIN) & (
    (_div.abs() / tbl_products['single_median_total'].clip(lower=1)) * 100 > XCHECK_DIVERGENCE_PCT)

tbl_products = tbl_products.sort_values('n_deliveries', ascending=False).reset_index(drop=True)
_show = tbl_products.head(25)[['product', 'install_min', 'ci_lo', 'ci_hi', 'n_deliveries',
                               'n_units', 'single_median_total', 'single_n',
                               'flag_zero_boundary', 'flag_xcheck']]
print(f'Base (per-visit) time: {BASE_MIN:.1f} min  [95% CI {ci_lo[0]:.1f}, {ci_hi[0]:.1f}]\n')
print(_show.to_string(index=False, float_format=lambda v: f'{v:,.1f}'))
print(f'\n  flagged at zero boundary (R6): {int(tbl_products["flag_zero_boundary"].sum())}   '
      f'cross-check divergent (R8): {int(tbl_products["flag_xcheck"].sum())}')

# Chart: top-20 products by support — one hue (identity is the y-label, not a color job),
# thin bars, CI whiskers, recessive grid.
_top = tbl_products[tbl_products['product_norm'] != OTHER_LABEL].head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(_top['product'].str.slice(0, 40), _top['install_min'],
        xerr=[_top['install_min'] - _top['ci_lo'], _top['ci_hi'] - _top['install_min']],
        color=PALETTE['primary'], height=0.62, error_kw=dict(ecolor='#555555', lw=1))
ax.set_xlabel('modelled install minutes per unit (95% bootstrap CI)')
ax.set_title(f'Install-time standards — top 20 products by support ({EFFECTIVE_START} to {EFFECTIVE_END})')
ax.grid(axis='x', color='#dddddd', lw=0.6); ax.set_axisbelow(True)
for s in ('top', 'right'): ax.spines[s].set_visible(False)
save_fig(fig, 'product_install_minutes'); plt.show()

## Cell 13 — REBUILD EXPECTED TICKET TIMES

Every delivery (including QC-excluded ones, so the export is complete) gets
`expected_min = base + Σ units × install_min`. The ratio actual/expected is the
workload-honest efficiency lens: a 90-minute ticket is not slow if it carried 85 minutes
of product.

*Teaching note:* distributions, not means — a P25/median/P75 of the ratio by warehouse
shows whether a site is uniformly slower or just has a heavy tail, and those are different
conversations.

In [ ]:
qty_all = (df_lines.groupby(['delivery_key', 'product_norm']).size().unstack(fill_value=0))
Xa = qty_all.reindex(columns=modeled_products, fill_value=0).copy()
_rare_cols = [c for c in qty_all.columns if c in rare_products]
Xa[OTHER_LABEL] = qty_all[_rare_cols].sum(axis=1) if _rare_cols else 0
expected = BASE_MIN + Xa.to_numpy(dtype=float) @ coef[1:]
df_dlv['expected_min'] = df_dlv['delivery_key'].map(pd.Series(expected, index=Xa.index))
df_dlv['residual_min'] = df_dlv['duration_min'] - df_dlv['expected_min']
df_dlv['ratio'] = df_dlv['duration_min'] / df_dlv['expected_min']

_ok = df_dlv[df_dlv['qc'] == 'ok']
print('actual/expected ratio (model sample): '
      + ', '.join(f'P{int(q*100)}={_ok["ratio"].quantile(q):.2f}' for q in (0.25, 0.50, 0.75)))
tbl_wh = (_ok.groupby('warehouse')
          .agg(deliveries=('delivery_key', 'nunique'),
               actual_median=('duration_min', 'median'),
               expected_median=('expected_min', 'median'),
               ratio_p25=('ratio', lambda s: s.quantile(.25)),
               ratio_median=('ratio', 'median'),
               ratio_p75=('ratio', lambda s: s.quantile(.75)))
          .sort_values('deliveries', ascending=False))
print('\nBy warehouse (proximity, not fault — coaching lens only, never discipline):')
print(tbl_wh.to_string(float_format=lambda v: f'{v:,.2f}'))

# Chart 1: actual vs expected (hexbin — a scatter of this size is an ink blot).
fig, ax = plt.subplots(figsize=(7, 6))
_lim = float(np.nanpercentile(_ok[['duration_min', 'expected_min']].to_numpy(), 99))
hb = ax.hexbin(_ok['expected_min'], _ok['duration_min'], gridsize=45, cmap='Blues',
               extent=(0, _lim, 0, _lim), mincnt=1)
ax.plot([0, _lim], [0, _lim], color='#555555', lw=1, ls='--', label='actual = expected')
ax.set_xlabel('expected minutes (model)'); ax.set_ylabel('actual minutes')
ax.set_title('Actual vs expected delivery time (model sample)')
ax.legend(frameon=False); fig.colorbar(hb, ax=ax, label='deliveries')
save_fig(fig, 'actual_vs_expected'); plt.show()

# Chart 2: ratio distribution — one series, one hue, reference line at 1.
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(_ok['ratio'].clip(0, 5), bins=100, color=PALETTE['primary'])
ax.axvline(1.0, color='#555555', lw=1, ls='--')
ax.set_xlabel('actual / expected'); ax.set_ylabel('deliveries')
ax.set_title('Where tickets land against the modelled standard')
ax.grid(axis='y', color='#dddddd', lw=0.6); ax.set_axisbelow(True)
for s in ('top', 'right'): ax.spines[s].set_visible(False)
save_fig(fig, 'ratio_distribution'); plt.show()

## Cell 14 — REASONABLENESS EVALUATION

Structured pass/attention verdicts the analyst reads before trusting a single number.
Each check states what "reasonable" means and prints which side of it this run landed on.

In [ ]:
checks = []
def _verdict(name, ok, detail):
    checks.append((name, ok, detail))
    print(f'  {"PASS " if ok else "LOOK "} {name}: {detail}')

print('REASONABLENESS CHECKS')
_verdict('fit explains real variance', R2 >= 0.15,
         f'R2={R2:.3f} (product mix should explain a visible share; near 0 means '
         f'timestamps do not measure install work)')
_verdict('median ratio centred', 0.85 <= _ok['ratio'].median() <= 1.15,
         f'median actual/expected={_ok["ratio"].median():.2f} (a calibrated standard sits near 1)')
_verdict('base time plausible', 5 <= BASE_MIN <= 60,
         f'per-visit base={BASE_MIN:.1f} min (paperwork+parking+greeting; <5 or >60 is suspect)')
_verdict('typical error tolerable', MAE <= _ok['duration_min'].median(),
         f'MAE={MAE:.1f} min vs median visit {_ok["duration_min"].median():.1f} min')
_share_25 = float(_ok['ratio'].between(0.75, 1.25).mean() * 100)
_verdict('mass near the standard', _share_25 >= 40,
         f'{_share_25:.1f}% of deliveries within +/-25% of expected')
_zero_share = float((tbl_products['install_min'] < 0.5).mean() * 100)
_verdict('few zero-boundary products', _zero_share <= 25,
         f'{_zero_share:.1f}% of modelled products priced ~0 min (R6 if high)')
_verdict('OTHER bucket is small', _rare_share <= 15,
         f'{_rare_share:.1f}% of units carry the pooled OTHER rate, not their own')
_verdict('exclusions are the minority', N_OK / len(df_dlv) >= 0.80,
         f'{N_OK/len(df_dlv)*100:.1f}% of deliveries survived QC')
_verdict('feed parses cleanly', UNPARSEABLE_COMPLETION_ROWS == 0,
         f'{UNPARSEABLE_COMPLETION_ROWS:,} lines invisible to the window (R1)')
_verdict('requested window served', not WINDOW_FELL_BACK,
         f'ran {EFFECTIVE_START} -> {EFFECTIVE_END}'
         + ('' if not WINDOW_FELL_BACK else
            f' INSTEAD OF the requested {FILTER_START} -> {FILTER_END} — the table lags (R0)'))
N_LOOK = sum(1 for _, ok, _ in checks if not ok)
print(f'\n{len(checks)-N_LOOK}/{len(checks)} checks passed; {N_LOOK} need a look '
      f'-> each maps to a research-queue category in Cell 15.')

## Cell 15 — ANOMALY & RESEARCH QUEUE

The standing structure for "items that need additional analysis". Every anomaly class gets
a category code, the affected records are **exported** (order numbers live only in the
OneDrive export, never in notebook output), and the queue carries `hypothesis / status /
owner / next_step` columns so research is tracked, not re-discovered.

| Code | Population | First question to ask |
|------|------------|----------------------|
| R0 | requested window empty — table lags | where do current-year orders live? |
| R1 | lines with unparseable timestamps | which feed/device writes these? |
| R2 | nonpositive durations | are arrival times back-filled at close-out? |
| R3 | durations > cap | multi-stop routes booked as one ticket? |
| R4 | split / multi-warehouse deliveries | should these be separate tickets? |
| R5 | bulk timestamp clusters | which admin process stamps these, and when? |
| R6 | products priced ~0 min | truly free-riding items, or collinearity victims? |
| R7 | below-support products in OTHER | any high-cost items worth a manual study? |
| R8 | model vs single-delivery median divergence | which estimate matches a ride-along? |
| R9 | tickets slower than 2.5x / faster than 0.4x expected | ticket-level review |

In [ ]:
_research = []
def _queue(cat, frame, key_col, detail_cols, note):
    f = frame.head(TOP_N_RESEARCH).copy()
    for _, r in f.iterrows():
        _research.append({'category': cat, 'key': str(r[key_col]),
                          'warehouse': r.get('warehouse', ''),
                          'detail': '; '.join(f'{c}={r[c]}' for c in detail_cols if c in r),
                          'note': note, 'hypothesis': '', 'status': 'open',
                          'owner': '', 'next_step': ''})
    print(f'  {cat}: {len(frame):,} in population ({min(len(frame), TOP_N_RESEARCH):,} queued)')

print('RESEARCH QUEUE POPULATIONS')
if WINDOW_FELL_BACK:
    _research.append({'category': 'R0', 'key': 'DATA GAP', 'warehouse': '',
                      'detail': f'SERP_ORDERS_HISTORY ends {EFFECTIVE_END}; requested '
                                f'{FILTER_START} -> {FILTER_END} is empty',
                      'note': 'current-year source table unknown', 'hypothesis': '',
                      'status': 'open', 'owner': '',
                      'next_step': 'ask IT which table carries current-year delivery orders'})
    print('  R0: requested window empty — queued as one data-gap item')
if UNPARSEABLE_COMPLETION_ROWS:
    _research.append({'category': 'R1', 'key': 'FEED', 'warehouse': '',
                      'detail': f'{UNPARSEABLE_COMPLETION_ROWS} lines with unparseable Completion_Time',
                      'note': 'upstream feed fix', 'hypothesis': '', 'status': 'open',
                      'owner': '', 'next_step': 'pull raw samples with IT'})
    print(f'  R1: {UNPARSEABLE_COMPLETION_ROWS:,} lines (queued as one feed item)')
_queue('R2', df_dlv[df_dlv['qc'] == 'nonpositive_duration'], 'delivery_key',
       ['duration_min', 'n_lines'], 'nonpositive duration')
_queue('R3', df_dlv[df_dlv['qc'] == 'above_max_duration'].sort_values('duration_min', ascending=False),
       'delivery_key', ['duration_min', 'n_lines'], 'exceeds duration cap')
_queue('R4', df_dlv[df_dlv['qc'].isin(['split_delivery', 'multi_warehouse'])]
       .sort_values('completion_spread_min', ascending=False), 'delivery_key',
       ['completion_spread_min', 'n_warehouses'], 'split/multi-warehouse')
_queue('R5', df_dlv[df_dlv['qc'] == 'bulk_timestamp_cluster'], 'delivery_key',
       ['completion_dt', 'n_lines'], 'bulk timestamp cluster')
_queue('R6', tbl_products[tbl_products['flag_zero_boundary'] &
       (tbl_products['product_norm'] != OTHER_LABEL)], 'product', ['install_min', 'n_deliveries'],
       'zero-boundary install time')
_r7 = (support[support < MIN_DELIVERIES_PER_PRODUCT].rename('n_deliveries').reset_index()
       .rename(columns={'product_norm': 'product'}).sort_values('n_deliveries', ascending=False))
_queue('R7', _r7, 'product', ['n_deliveries'], 'below support floor (in OTHER)')
_queue('R8', tbl_products[tbl_products['flag_xcheck']], 'product',
       ['install_min', 'single_median_total', 'xcheck_gap_min'], 'cross-check divergence')
_ok_r = df_dlv[df_dlv['qc'] == 'ok']
_queue('R9-slow', _ok_r[_ok_r['ratio'] > SLOW_RATIO].sort_values('ratio', ascending=False),
       'delivery_key', ['duration_min', 'expected_min', 'ratio'], 'far slower than standard')
_queue('R9-fast', _ok_r[_ok_r['ratio'] < FAST_RATIO].sort_values('ratio'),
       'delivery_key', ['duration_min', 'expected_min', 'ratio'], 'implausibly faster than standard')

df_research = pd.DataFrame(_research)
print(f'\n  research queue: {len(df_research):,} rows across '
      f'{df_research["category"].nunique() if len(df_research) else 0} categories '
      f'(detail in the OneDrive export only)')

## Cell 16 — EXPORTS

One workbook: product standards, every ticket's expected-vs-actual, the warehouse lens,
the exclusion ledger, and the research queue. Order numbers appear **here only** — the
workbook lands in the analyst's OneDrive report folder, outside the repository.

In [ ]:
_xlsx = os.path.join(OUT_DIR, f'install_time_model_{RUN_DATE}.xlsx')
_ledger_df = (df_dlv['qc'].value_counts().rename_axis('qc_reason')
              .reset_index(name='deliveries'))
_export_cols = ['order_num', 'warehouse', 'n_lines', 'n_products', 'arrival_dt',
                'completion_dt', 'duration_min', 'expected_min', 'residual_min', 'ratio', 'qc']
with pd.ExcelWriter(_xlsx, engine='openpyxl') as xw:
    tbl_products.drop(columns=['product_norm']).to_excel(xw, sheet_name='Product standards', index=False)
    df_dlv[_export_cols].to_excel(xw, sheet_name='Tickets expected vs actual', index=False)
    tbl_wh.reset_index().to_excel(xw, sheet_name='By warehouse', index=False)
    _ledger_df.to_excel(xw, sheet_name='Exclusion ledger', index=False)
    df_research.to_excel(xw, sheet_name='Research queue', index=False)
print(f'Wrote {_xlsx}')
print('Charts saved alongside: product_install_minutes, actual_vs_expected, ratio_distribution.')
print('\nCAVEAT (goes with every use of this file): expected times are modeled averages —')
print('proximity, not fault. Planning and coaching only; never discipline without')
print('ticket-level review.')